In [1]:
import numpy as np
import pandas as pd


In [2]:
df = pd.read_csv('/content/drive/MyDrive/Finance_Club_Project/Dataset/processed_data.csv')

In [3]:
df.head()

,Customer_ID,marriage,sex,education,LIMIT_BAL,age,pay_0,pay_2,pay_3,pay_4,...,total_payment_gap,payment_std,payment_mean,payment_cv,weighted_pay_score,bill_trend,pay_trend,age_to_limit,marriage_education,next_month_default
0,5017,2,0,2,60000,25.0,2,2,2,0,...,241111.88,714.956931,1326.185000,0.539108,30,-9752.770000,76.776286,0.000417,22.0,0
1,5018,2,1,1,290000,24.0,0,0,-2,-2,...,11160.47,1129.473118,674.998333,1.673298,-16,-1169.548571,269.948286,0.000083,21.0,0
2,5019,1,0,2,180000,63.0,0,0,0,0,...,290474.38,123.925072,2009.286667,0.061676,0,998.838286,-31.773714,0.000350,12.0,0
3,5020,1,1,2,210000,43.0,0,0,0,0,...,498929.03,337.634028,3074.916667,0.109803,0,-3253.982000,-140.201714,0.000205,12.0,0
4,5021,2,0,1,280000,32.0,-2,-2,-2,-2,...,19571.32,17912.037141,8552.366667,2.094395,-42,-1522.201714,-1642.400571,0.000114,21.0,0


In [4]:
selected_features = [
    'mean_delinquency', 'weighted_pay_score', 'marriage_education', 'max_delinquency',
    'pay_0', 'credit_utilization', 'pay_2', 'Bill_amt1',
    'payment_mean', 'AVG_Bill_amt', 'delinquent_months', 'pay_amt2',
    'pay_amt1', 'pay_3', 'age_to_limit', 'Bill_amt2',
    'payment_cv', 'pay_amt3', 'bill_trend', 'payment_std'
]

In [5]:
from sklearn.model_selection import train_test_split

target = 'next_month_default'

X = df.drop(columns=[target,'Customer_ID'])

y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train = X_train[selected_features]
X_test = X_test[selected_features]

In [40]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

# Model Training

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score
)

results = []
def train_model(model, X_train, y_train, X_test, y_test,selected_features, model_name):
    """
    Train and evaluate a classification model.

    Parameters
    ----------
    model : Machine learning model
    X_train : Training features
    y_train : Training labels
    X_test : Testing features
    y_test : Testing labels

    Returns
    -------
    dict : Model name and evaluation metrics
    """



    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    print('Model Name :', model_name)
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision :", precision_score(y_test, y_pred, average='weighted'))
    print("Recall :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))
    print("F2 Score :", fbeta_score(y_test, y_pred, average='weighted', beta=2))
    print("ROC AUC Score :", roc_auc_score(y_test, y_pred))

    result = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1_score': f1_score(y_test, y_pred, average='weighted'),
        'f2_score': fbeta_score(y_test, y_pred, average='weighted', beta=2),
        'roc_auc_score': roc_auc_score(y_test, y_pred)
    }


    results.append(result)
    return result





In [8]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

lr= LogisticRegression(max_iter=10000,random_state=42)
lr_results = train_model(lr, X_train, y_train, X_test, y_test,selected_features, "LogisticRegression")

Model Name : LogisticRegression
Accuracy : 0.7145303326810176
Precision : 0.7204433478277115
Recall : 0.7145303326810176
F1 Score : 0.712806188177355
F2 Score : 0.7127392896345216
ROC AUC Score : 0.7149538694269612


In [9]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt_results = train_model(dt, X_train, y_train, X_test, y_test,selected_features,"DecisionTree")

Model Name : DecisionTree
Accuracy : 0.8217954990215264
Precision : 0.8218759521160002
Recall : 0.8217954990215264
F1 Score : 0.8217737279892453
F2 Score : 0.8217719170934892
ROC AUC Score : 0.8217436028163047


In [10]:
# Randam Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf_results = train_model(rf, X_train, y_train, X_test, y_test,selected_features,"RandamForest")

Model Name : RandamForest
Accuracy : 0.8875978473581213
Precision : 0.8893477671702096
Recall : 0.8875978473581213
F1 Score : 0.8874943276792416
F2 Score : 0.8873224174383216
ROC AUC Score : 0.8877729746290824


In [11]:
# XGBoost
import xgboost as xgb

xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_results = train_model(xgb, X_train, y_train, X_test, y_test,selected_features,"XGBoost")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:32:06] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model Name : XGBoost
Accuracy : 0.8789138943248532
Precision : 0.88342074280446
Recall : 0.8789138943248532
F1 Score : 0.8785975351640289
F2 Score : 0.8781765398023054
ROC AUC Score : 0.8792000124466977


In [12]:
# LightGBM
import lightgbm as lgb

lgb = lgb.LGBMClassifier(random_state=42)
lgb_results = train_model(lgb, X_train, y_train, X_test, y_test,selected_features,"LightGBM")

[LightGBM] [Info] Number of positive: 16330, number of negative: 16374
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002093 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3764
[LightGBM] [Info] Number of data points in the train set: 32704, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499327 -> initscore=-0.002691
[LightGBM] [Info] Start training from score -0.002691
Model Name : LightGBM
Accuracy : 0.8827054794520548
Precision : 0.887876922098163
Recall : 0.8827054794520548
F1 Score : 0.8823547818368782
F2 Score : 0.8818682277527576
ROC AUC Score : 0.8830107963133839


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
model_comparison_df = pd.DataFrame(results)
model_comparison_df = model_comparison_df.sort_values(by='f1_score', ascending=False).reset_index(drop=True)
model_comparison_df

,model_name,accuracy,precision,recall,f1_score,f2_score,roc_auc_score
0,RandamForest,0.887598,0.889348,0.887598,0.887494,0.887322,0.887773
1,LightGBM,0.882705,0.887877,0.882705,0.882355,0.881868,0.883011
2,XGBoost,0.878914,0.883421,0.878914,0.878598,0.878177,0.879200
3,DecisionTree,0.821795,0.821876,0.821795,0.821774,0.821772,0.821744
4,LogisticRegression,0.714530,0.720443,0.714530,0.712806,0.712739,0.714954


# Models Tunning

In [14]:

def evaluate_after_model_tunning(model, X_train, y_train, X_test, y_test,selected_features,model_name):
    """
    Train and evaluate a classification model.

    Parameters
    ----------
    model : Machine learning model
    X_train : Training features
    y_train : Training labels
    X_test : Testing features
    y_test : Testing labels

    Returns
    -------
    dict : Model name and evaluation metrics
    """


    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    print('Model Name :', model_name)
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision :", precision_score(y_test, y_pred, average='weighted'))
    print("Recall :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))
    print("F2 Score :", fbeta_score(y_test, y_pred, average='weighted', beta=2))
    print("ROC AUC Score :", roc_auc_score(y_test, y_pred))

    optimized_result = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1_score': f1_score(y_test, y_pred, average='weighted'),
        'f2_score': fbeta_score(y_test, y_pred, average='weighted', beta=2),
        'roc_auc_score': roc_auc_score(y_test, y_pred)
    }



    return optimized_result

In [15]:
# LogisticRegression

from sklearn.model_selection import GridSearchCV

lr_param = { 'C' : [ 1, 10, 20, 100,] }
lr_grid = GridSearchCV(LogisticRegression(max_iter=10000), lr_param, cv=5, scoring='f1')
lr_grid.fit(X_train, y_train)
lr_grid_results = evaluate_after_model_tunning(lr_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features, "LogisticRegression")

print("Best Parameter :", lr_grid.best_params_)



Model Name : LogisticRegression
Accuracy : 0.7145303326810176
Precision : 0.720407203251628
Recall : 0.7145303326810176
F1 Score : 0.712817124986125
F2 Score : 0.712750353245348
ROC AUC Score : 0.7149525529493288
Best Parameter : {'C': 10}


In [16]:
# Decission Tree

dt_param = {
    'max_depth': [None, 10,20,25,30]
}

dt_grid = GridSearchCV(DecisionTreeClassifier(), dt_param, cv=5, scoring='f1')
dt_grid.fit(X_train, y_train)
dt_grid_results = evaluate_after_model_tunning(dt_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"DecisionTree")

print("Best Parameter :", dt_grid.best_params_)

Model Name : DecisionTree
Accuracy : 0.8282778864970646
Precision : 0.8321858662075989
Recall : 0.8282778864970646
F1 Score : 0.8278335371882682
F2 Score : 0.827529869389127
ROC AUC Score : 0.828561221595499
Best Parameter : {'max_depth': 10}


In [17]:
# Random Forest

rf_param = {
    'n_estimators': [100, 125, 150],
    'max_depth': [None, 5,10]
}

rf_grid = GridSearchCV(RandomForestClassifier(), rf_param, cv=5, scoring = 'f1')
rf_grid.fit(X_train, y_train)
rf_grid_results = evaluate_after_model_tunning(rf_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"RandomForest")

print("Best Parameter :", rf_grid.best_params_)

Model Name : RandomForest
Accuracy : 0.8883317025440313
Precision : 0.8898852248622157
Recall : 0.8883317025440313
F1 Score : 0.8882417610479498
F2 Score : 0.8880883490034155
ROC AUC Score : 0.8884963192482196
Best Parameter : {'max_depth': None, 'n_estimators': 125}


In [24]:
# XGBoost
from xgboost import XGBClassifier

xgb_param = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1]
}

xgb_grid = GridSearchCV(XGBClassifier( eval_metric='logloss', random_state=42),xgb_param, cv=5, scoring='f1')
xgb_grid.fit(X_train, y_train)
xgb_grid_results = evaluate_after_model_tunning(xgb_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"XGBoost")

print("Best Parameter :", xgb_grid.best_params_)

Model Name : XGBoost
Accuracy : 0.8792808219178082
Precision : 0.8841913836258842
Recall : 0.8792808219178082
F1 Score : 0.8789359358508623
F2 Score : 0.8784778957287308
ROC AUC Score : 0.8795794572043041
Best Parameter : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}


In [30]:
# LightGBM
from lightgbm import LGBMClassifier

lgb_param = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1]

}

lgb_grid = GridSearchCV(LGBMClassifier(random_state=42), lgb_param, cv=5)
lgb_grid.fit(X_train, y_train)
lgb_grid_results = evaluate_after_model_tunning(lgb_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"LightGBM")

print("Best Parameter :", lgb_grid.best_params_)

[LightGBM] [Info] Number of positive: 13064, number of negative: 13099
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001833 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3759
[LightGBM] [Info] Number of data points in the train set: 26163, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499331 -> initscore=-0.002676
[LightGBM] [Info] Start training from score -0.002676
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 13064, number of negative: 13099
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001660 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3759
[LightGBM] [Info] Number of data points in the train set: 26163, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499331 -> initscore=-0.002676
[LightGBM] [Info] Start training from score -0.002676
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 13064, number of negative: 13099
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001662 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3762
[LightGBM] [Info] Number of data points in the train set: 26163, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499331 -> initscore=-0.002676
[LightGBM] [Info] Start training from score -0.002676
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 13064, number of negative: 13100
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003062 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3762
[LightGBM] [Info] Number of data points in the train set: 26164, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499312 -> initscore=-0.002752
[LightGBM] [Info] Start training from score -0.002752
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 16330, number of negative: 16374
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003699 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3764
[LightGBM] [Info] Number of data points in the train set: 32704, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499327 -> initscore=-0.002691
[LightGBM] [Info] Start training from score -0.002691
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# Final Comparision

In [31]:
optimized_results = []

optimized_results.append(evaluate_after_model_tunning(lr_grid.best_estimator_,X_train, y_train, X_test, y_test,selected_features, "LogisticRegression"))
optimized_results.append(evaluate_after_model_tunning(dt_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"DecisionTree"))
optimized_results.append(evaluate_after_model_tunning(rf_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"RandomForest"))
optimized_results.append(evaluate_after_model_tunning(xgb_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"XGBoost"))
optimized_results.append(evaluate_after_model_tunning(lgb_grid.best_estimator_, X_train, y_train, X_test, y_test,selected_features,"LightGBM"))


optimized_model_comparison_df = pd.DataFrame(optimized_results)
optimized_model_comparison_df = optimized_model_comparison_df.sort_values(by='f1_score', ascending=False).reset_index(drop=True)

Model Name : LogisticRegression
Accuracy : 0.7145303326810176
Precision : 0.720407203251628
Recall : 0.7145303326810176
F1 Score : 0.712817124986125
F2 Score : 0.712750353245348
ROC AUC Score : 0.7149525529493288
Model Name : DecisionTree
Accuracy : 0.8291340508806262
Precision : 0.8331052249289342
Recall : 0.8291340508806262
F1 Score : 0.8286854831535705
F2 Score : 0.8283759108898882
ROC AUC Score : 0.8294193854921771
Model Name : RandomForest
Accuracy : 0.886619373776908
Precision : 0.8883145564986209
Recall : 0.886619373776908
F1 Score : 0.8865183097934485
F2 Score : 0.8863520338939337
ROC AUC Score : 0.8867918397535554
Model Name : XGBoost
Accuracy : 0.8792808219178082
Precision : 0.8841913836258842
Recall : 0.8792808219178082
F1 Score : 0.8789359358508623
F2 Score : 0.8784778957287308
ROC AUC Score : 0.8795794572043041
[LightGBM] [Info] Number of positive: 16330, number of negative: 16374
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.00903

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [32]:
model_comparison_df

,model_name,accuracy,precision,recall,f1_score,f2_score,roc_auc_score
0,RandamForest,0.887598,0.889348,0.887598,0.887494,0.887322,0.887773
1,LightGBM,0.882705,0.887877,0.882705,0.882355,0.881868,0.883011
2,XGBoost,0.878914,0.883421,0.878914,0.878598,0.878177,0.879200
3,DecisionTree,0.821795,0.821876,0.821795,0.821774,0.821772,0.821744
4,LogisticRegression,0.714530,0.720443,0.714530,0.712806,0.712739,0.714954


# Prediction

In [33]:
Validation_data= pd.read_csv('/content/drive/MyDrive/Finance_Club_Project/Dataset/validate_dataset_final.csv')

Validation_data.head()

,Customer_ID,marriage,sex,education,LIMIT_BAL,age,pay_0,pay_2,pay_3,pay_4,...,Bill_amt5,Bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,AVG_Bill_amt,PAY_TO_BILL_ratio
0,1,1,1,2,220000,32,0,0,0,0,...,17831.13,15670.47,2000.03,3999.90,1419.80,1999.97,3000.21,30788.71,23456.33,0.31
1,2,2,0,1,350000,35,-1,-1,-1,0,...,10832.78,2261.45,33891.01,16267.19,4026.80,234.10,1565.11,26346.39,13660.33,1.00
2,3,2,1,1,310000,39,0,0,0,0,...,240520.57,246524.45,11026.94,10499.83,14000.32,10000.12,10000.16,10000.21,258034.17,0.04
3,4,1,0,2,20000,47,0,0,0,2,...,15040.17,14749.97,1200.00,2799.83,0.14,1499.93,0.02,1000.26,13333.33,0.08
4,5,2,1,2,500000,30,0,0,0,0,...,69054.15,64841.30,25463.94,43095.31,7521.96,9065.17,8841.07,12000.21,77204.50,0.23


In [34]:
vdf=Validation_data.copy()

In [35]:

# CREDIT UTILIZATION RATIO
# Measures how much credit the customer is using compared to the limit.
vdf['credit_utilization'] = vdf['AVG_Bill_amt'] / (vdf['LIMIT_BAL'] + 1e-6)


# DELINQUENCY FEATURES
# How many months a customer was late in repayment.
pay_cols = ['pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']
vdf['delinquent_months'] = vdf[pay_cols].apply(lambda row: sum(row >= 1), axis=1)
vdf['max_delinquency'] = vdf[pay_cols].max(axis=1)
vdf['mean_delinquency'] = vdf[pay_cols].mean(axis=1)


# REPAYMENT GAP FEATURES
# Difference between bill and paid amount for each month.
for i in range(1, 7):
    vdf[f'diff_amt_{i}'] = vdf[f'Bill_amt{i}'] - vdf[f'pay_amt{i}']


# Summary statistics for repayment gap
vdf['avg_payment_gap'] = vdf[[f'diff_amt_{i}' for i in range(1, 7)]].mean(axis=1)
vdf['total_payment_gap'] = vdf[[f'diff_amt_{i}' for i in range(1, 7)]].sum(axis=1)


# PAYMENT CONSISTENCY
pay_amt_cols = [f'pay_amt{i}' for i in range(1, 7)]
vdf['payment_std'] = vdf[pay_amt_cols].std(axis=1)
vdf['payment_mean'] = vdf[pay_amt_cols].mean(axis=1)
vdf['payment_cv'] = vdf['payment_std'] / (df['payment_mean'] + 1e-6)


# RECENT-WEIGHTED DELINQUENCY SCORE
pay_cols_weighted = [f'pay_{i}' for i in [0, 2, 3, 4, 5, 6]]
weights = [6, 5, 4, 3, 2, 1]
vdf['weighted_pay_score'] = vdf[pay_cols_weighted].mul(weights, axis=1).sum(axis=1)


# BILL & PAYMENT TRENDS
from scipy.stats import linregress

def get_slope(row):
    return linregress(range(1, 7), row).slope

vdf['bill_trend'] = vdf[[f'Bill_amt{i}' for i in range(1, 7)]].apply(get_slope, axis=1)
vdf['pay_trend'] = vdf[[f'pay_amt{i}' for i in range(1, 7)]].apply(get_slope, axis=1)


# DEMOGRAPHIC RISK INTERACTIONS
vdf['age_to_limit'] = vdf['age'] / (vdf['LIMIT_BAL'] + 1e-6)
vdf['marriage_education'] = vdf['marriage'].astype(str) + '_' + vdf['education'].astype(str)


# OPTIONAL CLEAN-UP
diff_cols = [f'diff_amt_{i}' for i in range(1, 7)]
vdf.drop(columns=diff_cols, inplace=True)


vdf.head()

,Customer_ID,marriage,sex,education,LIMIT_BAL,age,pay_0,pay_2,pay_3,pay_4,...,avg_payment_gap,total_payment_gap,payment_std,payment_mean,payment_cv,weighted_pay_score,bill_trend,pay_trend,age_to_limit,marriage_education
0,1,1,1,2,220000,32,0,0,0,0,...,16254.906667,97529.44,11591.348629,7201.436667,8.740371,0,-2998.849143,4043.557143,0.000145,1_2
1,2,2,0,1,350000,35,-1,-1,-1,0,...,-61.606667,-369.64,14116.320363,13721.766667,20.913119,-15,-2211.464571,-2446.344000,0.000100,2_1
2,3,2,1,1,310000,39,0,0,0,0,...,247112.998333,1482677.99,1562.986833,10921.263333,0.777881,0,-9761.167714,-303.796000,0.000126,2_1
3,4,1,0,2,20000,47,0,0,0,2,...,12249.880000,73499.28,1047.597389,1083.363333,0.340691,12,862.914857,-225.666857,0.002350,1_2
4,5,2,1,2,500000,30,0,0,0,0,...,59540.101667,357240.61,14101.872597,17664.610000,1.648885,0,-5552.626000,-4815.376000,0.000060,2_2


In [42]:
# 1. Same selected features
selected_features = [
    'mean_delinquency', 'weighted_pay_score', 'marriage_education', 'max_delinquency',
    'pay_0', 'credit_utilization', 'pay_2', 'Bill_amt1',
    'payment_mean', 'AVG_Bill_amt', 'delinquent_months', 'pay_amt2',
    'pay_amt1', 'pay_3', 'age_to_limit', 'Bill_amt2',
    'payment_cv', 'pay_amt3', 'bill_trend', 'payment_std'
]



# 2. Feature selection from validation data (vdf)
X_val = vdf[selected_features]


# 3. Apply same StandardScaler (trained on training data)
X_val_scaled = scale.transform(X_val)


# 4. Predict using best model
y_val_pred = rf_grid.best_estimator_.predict(X_val_scaled)


# 5. Add predictions to DataFrame
vdf['next_month_default'] = y_val_pred


# 6. Save to CSV for submission
vdf[['next_month_default']].to_csv('/content/drive/MyDrive/Finance_Club_Project/submission.csv', index=False)

print("✅ Submission file saved as 'submission.csv'")

✅ Submission file saved as 'submission.csv'


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
